<div style="float: block; text-align: center; line-height: 1.7em">
    <span style="font-size: 2em; font-weight: bold"> Fatigue-Sleepiness in Irregular Workloads for Pilots </span><br>
    <span style="font-size: 1.5em; font-weight: bold"> Statistical Modelling - Perceived Excessive Sleepiness </span><br>
</div>

----

---

# 1. Loading Libraries and Packages

In [1]:
import os                                            # system library
import pandas as pd                                  # library to handle databases
import numpy as np                                   # library to handle arrays and matrices
import matplotlib.pyplot as plt                      # graphical package
import seaborn as sns                                # beautification graphical package
from IPython.display import display, HTML

os.environ['R_HOME'] = "C:/PROGRA~1/R/R-44~1.2"      # Necessary to configure R_HOME path

import rpy2.robjects as ro                           # Forcing to UTF-8 locale
ro.r('Sys.setlocale("LC_ALL", "English_United States.UTF-8")')

from pymer4.models import Lmer                       # Package for statistical modelling

----

# 2. Reading Pre-Processed Data

In [3]:
file = os.path.join('data','slp_wrk_mdl_train.csv')
try:
    df = pd.read_csv(file)
    print('-----------------------------------------------')
    print(f"\033[93mSuccess Data file read!\033[0m")
    print('-----------------------------------------------')
    display(df.head(3))
except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

-----------------------------------------------
Success Data file read!
-----------------------------------------------


,Id,TPFS_slp,duty_moment,DMod_slp,sleep_dev,Sleep_quality,time_awake,sleep_duration,age,kssd
0,P01,base,start,base,0,5,8.466667,8.383333,base,0
1,P01,em+eve,middle,base,0,5,8.466667,8.383333,base,0
2,P01,NI,end,base,0,5,8.466667,8.383333,base,1


----

# 3. Verifying for Null rows

In [4]:
df['Sleep_quality'] = pd.to_numeric(df['Sleep_quality'], errors = 'coerce')
wsm = df.isna().sum()
display(pd.DataFrame(wsm).T)

,Id,TPFS_slp,duty_moment,DMod_slp,sleep_dev,Sleep_quality,time_awake,sleep_duration,age,kssd
0,0,0,0,0,0,29,7,3,0,0


* As we can see above for covariates time_awake and sleep duration there are some null rows, so these rows containing null values will be taken away.

In [5]:
df = df.dropna()

wsm = df.isna().sum()
display(pd.DataFrame(wsm).T)

,Id,TPFS_slp,duty_moment,DMod_slp,sleep_dev,Sleep_quality,time_awake,sleep_duration,age,kssd
0,0,0,0,0,0,0,0,0,0,0


* Now as we can see, there is no longer null rows!

In [7]:
print(df.nunique())

Id                 48
TPFS_slp            3
duty_moment         3
DMod_slp            2
sleep_dev           2
Sleep_quality       9
time_awake        265
sleep_duration    250
age                 2
kssd                2
dtype: int64


---

# 4. Scalling some covariates

To avoid scalling bias in the model, we will rescalling some continuous covariates.

In [8]:
df[['Sleep_quality','time_awake','sleep_duration']].describe().T

,count,mean,std,min,25%,50%,75%,max
Sleep_quality,1046.0,7.293499,1.723159,2.000000,6.000000,7.500000,9.000000,10.000000
time_awake,1046.0,3.923470,3.592872,0.250000,1.500000,2.391667,5.150000,17.400000
sleep_duration,1046.0,7.183748,1.780239,2.466667,6.166667,7.200000,8.383333,12.416667


Above is shown some descriptive statistics of the ordinal and continuous covariates, based on the covariate statistics we propose the max-min standardization.

In [9]:
df['Sleep_quality'] = df['Sleep_quality']/(df['Sleep_quality'].max()-df['Sleep_quality'].min())
df['time_awake'] = df['time_awake']/(df['time_awake'].max()-df['time_awake'].min())
df['sleep_duration'] = df['sleep_duration']/(df['sleep_duration'].max()-df['sleep_duration'].min())

---

# 5. Preliminary Regression Analysis

Once we have a repeated measure design, we choose a mixed logistic model given by:

<p style="text-align: center;"> $g(x_{ij},\beta_{0i},\beta_1, \beta_s) = \beta_{0i}+\beta_{1i}x_1 + x_{ij}^T\beta_s$  (level 1) </p>
<p style="text-align: center;"> $\beta_{0i} = \beta_0 + \alpha_i$  (level 1, random intercept) </p>

where $\alpha_i \sim N(0,\sigma_\alpha^2)$ .

with $\alpha_i$ representing the random intercept due to participant variability and $\beta_0$ the intercept or baselines.

The logistic link function os given by:

<p style="text-align: center;"> $$g(x_{ij},\beta_{0i},\beta_1, \beta_s) = ln\left[\frac{\pi(x_{ij},\beta_{0i},\beta_1, \beta_s)}{1-\pi(x_{ij},\beta_{0i},\beta_1, \beta_s)}\right]$$ </p>

with, 

<p style="text-align: center;"> $$\pi(x_{ij},\beta_{0i},\beta_1, \beta_s) = \frac{e^{\beta_{0i}+\beta_{1i}x_1 + x_{ij}^T\beta_s}}{1+e^{\beta_{0i}+\beta_{1i}x_1 + x_{ij}^T\beta_s}}$$</p>


## 5.1. Function to codify the covariates in pymer standards

In [10]:
def get_formula(covariates, target, prnt = False):
    formula = f'{target}~'
    for cvt in covariates:
        formula += cvt+'+'
    formula = formula[:-1]
    if prnt:
        print(f'formula:\n{formula}')
    return formula

## 5.2. First Iteration

In [11]:
covariates = ['TPFS_slp','duty_moment','DMod_slp','sleep_dev','Sleep_quality','time_awake','sleep_duration','age']

formula = get_formula(covariates, 'kssd', prnt=False)
ff = formula + '+(1|Id)'

print('----------------------------------------------------------------------------------------------------------')
print(f'\033[91mFormula: {ff}\033[0m')
print('----------------------------------------------------------------------------------------------------------')


model_LME = Lmer(ff, data = df, family = 'binomial')
mdf = model_LME.fit()

print(mdf)

----------------------------------------------------------------------------------------------------------
Formula: kssd~TPFS_slp+duty_moment+DMod_slp+sleep_dev+Sleep_quality+time_awake+sleep_duration+age+(1|Id)
----------------------------------------------------------------------------------------------------------
Model failed to converge with max|grad| = 0.00818334 (tol = 0.002, component 1) 

Linear mixed model fit by maximum likelihood  ['lmerMod']
Formula: kssd~TPFS_slp+duty_moment+DMod_slp+sleep_dev+Sleep_quality+time_awake+sleep_duration+age+(1|Id)

Family: binomial	 Inference: parametric

Number of observations: 1046	 Groups: {'Id': 48.0}

Log-likelihood: -381.428 	 AIC: 786.857

Random effects:

           Name    Var    Std
Id  (Intercept)  0.453  0.673

No random effect correlations specified

Fixed effects:

                   Estimate  2.5_ci  97.5_ci     SE     OR  OR_2.5_ci  \
(Intercept)           0.947  -0.371    2.265  0.672  2.578      0.690   
TPFS_slpem+eve      

C:\Users\jlpsc\anaconda3\envs\pymer4_projects\lib\site-packages\pymer4\models\Lmer.py:733: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  ran_vars = ran_vars.applymap(


## 5.3. Second Iteration

In [14]:
covariates = ['TPFS_slp','duty_moment','DMod_slp','Sleep_quality','sleep_duration','age']

formula = get_formula(covariates, 'kssd', prnt=False)
ff = formula + '+(1|Id)'

print('----------------------------------------------------------------------------------------------------------')
print(f'\033[91mFormula: {ff}\033[0m')
print('----------------------------------------------------------------------------------------------------------')


model_LME = Lmer(ff, data = df, family = 'binomial')
mdf = model_LME.fit()

print(mdf)

----------------------------------------------------------------------------------------------------------
Formula: kssd~TPFS_slp+duty_moment+DMod_slp+Sleep_quality+sleep_duration+age+(1|Id)
----------------------------------------------------------------------------------------------------------
Linear mixed model fit by maximum likelihood  ['lmerMod']
Formula: kssd~TPFS_slp+duty_moment+DMod_slp+Sleep_quality+sleep_duration+age+(1|Id)

Family: binomial	 Inference: parametric

Number of observations: 1046	 Groups: {'Id': 48.0}

Log-likelihood: -382.747 	 AIC: 785.494

Random effects:

           Name    Var    Std
Id  (Intercept)  0.413  0.642

No random effect correlations specified

Fixed effects:

                   Estimate  2.5_ci  97.5_ci     SE     OR  OR_2.5_ci  \
(Intercept)           1.224   0.179    2.268  0.533  3.400      1.196   
TPFS_slpem+eve        1.181   0.728    1.633  0.231  3.256      2.072   
TPFS_slpNI            1.627   1.057    2.196  0.291  5.087      2.878  

C:\Users\jlpsc\anaconda3\envs\pymer4_projects\lib\site-packages\pymer4\models\Lmer.py:733: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  ran_vars = ran_vars.applymap(
